<a href="https://colab.research.google.com/github/PavanRishi69/dataviz-exercises-pavan-rishi-gillella/blob/main/lecture10_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**WEEK 10 Exercise **






In [1]:
!pip install -qqq streamlit

import streamlit as st
import pandas as pd
import plotly.express as px
from pathlib import Path

st.set_page_config(page_title="CO2 Dashboard", page_icon="🌱", layout="wide")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 29.8 MB/s eta 0:00:00


2026-07-23 16:22:46.152 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


── Data ──────────────────────────────────────────────────────────────────────
# @st.cache_data: Streamlit reruns the entire script on every widget interaction.
# Without caching, the CSV is read from disk on every interaction — slow and wasteful.
# cache_data stores the result after the first run and reuses it until the file changes.
@st.cache_data
def load_data():
    path = Path(__file__).parent.parent / 'data' / 'co2_emissions.csv'
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-01-01')
    return df

df = load_data()

st.title("🌱 CO2 Emissions Explorer")
st.caption("Source: Our World in Data — ourworldindata.org/co2-emissions")


In [4]:
import pandas as pd

# Assuming 'co2_emissions.csv' is directly accessible in the Colab environment.

def load_data():
    path = '/content/co2_emissions (1).csv'
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-01-01')
    return df

df = load_data()

In [5]:
# ── TASK 1: Sidebar with 5 widgets ────────────────────────────────────────────
#   a) st.selectbox for Region (with 'All')
#   b) st.multiselect for Countries (updates based on region — chained)
#   c) st.date_input for date range (two-handle; convert years to Jan-1 dates)
#   d) st.radio for Metric: "Total CO2 (Mt)" vs "CO2 per capita"
#   e) st.checkbox labelled "Show only top emitter highlighted"
#
# Guards:
#   - empty countries → st.warning + st.stop()
#   - incomplete date_input → st.warning + st.stop()
# Convert date_input result to pd.Timestamp before filtering.
# ─────────────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.header("Filters")

    # Region
    regions = ["All"] + sorted(df["Region"].dropna().unique().tolist())
    selected_region = st.selectbox("Region", regions)

    # Countries (chained)
    if selected_region == "All":
        country_list = sorted(df["Country"].unique())
    else:
        country_list = sorted(
            df[df["Region"] == selected_region]["Country"].unique()
        )

    selected_countries = st.multiselect(
        "Countries",
        country_list,
        default=country_list[:5]
    )

    if len(selected_countries) == 0:
        st.warning("Please select at least one country.")
        st.stop()

        # Date range
    min_date = df["Date"].min().date()
    max_date = df["Date"].max().date()

    date_range = st.date_input(
        "Date Range",
        value=(min_date, max_date),
        min_value=min_date,
        max_value=max_date
    )

    if len(date_range) != 2:
        st.warning("Please select a valid date range.")
        st.stop()

    start_date = pd.Timestamp(date_range[0])
    end_date = pd.Timestamp(date_range[1])

    # Metric
    metric = st.radio(
        "Metric",
        ["Total CO2 (Mt)", "CO2 per capita"]
    )

    # Highlight checkbox
    highlight = st.checkbox("Show only top emitter highlighted")



2026-07-23 16:27:53.755 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:27:53.927 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-07-23 16:27:53.928 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:27:53.929 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:27:53.932 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:27:53.936 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:27:53.937 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:27:53.938 Session state does not 

In [6]:


filtered = df[
    (df["Country"].isin(selected_countries)) &
    (df["Date"] >= start_date) &
    (df["Date"] <= end_date)
]

if selected_region != "All":
    filtered = filtered[filtered["Region"] == selected_region]

metric_col = "CO2_Mt" if metric == "Total CO2 (Mt)" else "CO2_per_capita"

In [7]:
# ── TASK 2: Filter summary caption ────────────────────────────────────────────
# Show: "X countries | Region | Date range | Metric"
# BBD rule: always show users how many records match current filters
# ─

st.caption(
    f"{len(selected_countries)} countries | "
    f"{selected_region} | "
    f"{start_date.year}-{end_date.year} | "
    f"{metric}"
)

2026-07-23 16:28:04.567 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:04.568 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:04.570 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [8]:
col_left, col_right = st.columns([2, 1])

with col_left:
    # Line chart
    if highlight:
        totals = (
            filtered.groupby("Country")[metric_col]
            .sum()
            .sort_values(ascending=False)
        )

        top_country = totals.index[0]

        filtered["Highlight"] = filtered["Country"].apply(
            lambda x: x if x == top_country else "Other"
        )

        fig = px.line(
            filtered,
            x="Date",
            y=metric_col,
            color="Highlight",
            line_group="Country",
            hover_name="Country",
            title="Trend of CO₂ Emissions",
            color_discrete_map={
                "Other": "lightgrey",
                top_country: "red"
            }
        )

    else:

        fig = px.line(
            filtered,
            x="Date",
            y=metric_col,
            color="Country",
            title="Trend of CO₂ Emissions"
        )

    fig.update_layout(
        template="plotly_white"
    )

    st.plotly_chart(fig, use_container_width=True)

with col_right:
    # Bar chart
    latest_year = filtered["Year"].max()

    latest = filtered[
        filtered["Year"] == latest_year
    ].sort_values(metric_col, ascending=False)

    fig2 = px.bar(
        latest,
        x=metric_col,
        y="Country",
        orientation="h",
        title=f"Ranking ({latest_year})"
    )

    fig2.update_layout(
        template="plotly_white",
        yaxis=dict(categoryorder="total ascending")
    )

    st.plotly_chart(fig2, use_container_width=True)

2026-07-23 16:28:08.198 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:08.199 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:08.201 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:09.823 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-07-23 16:28:09.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:09.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:09.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [10]:
# ── EXTENSION: KPI row above the charts ───────────────────────────────────────
#   - Total CO2 in last year of selected range (sum across selected countries)
#   - % change from first to last year
#   - Country with highest emissions in last year


last_year = filtered["Year"].max()
first_year = filtered["Year"].min()

last_data = filtered[filtered["Year"] == last_year]
first_data = filtered[filtered["Year"] == first_year]

total_last = last_data[metric_col].sum()
total_first = first_data[metric_col].sum()

if total_first != 0:
    pct_change = ((total_last - total_first) / total_first) * 100
else:
    pct_change = 0

top_country = last_data.sort_values(
    metric_col,
    ascending=False
).iloc[0]["Country"]

k1, k2, k3 = st.columns(3)

k1.metric(
    "Total CO₂",
    f"{total_last:,.2f}"
)

k2.metric(
    "% Change",
    f"{pct_change:.2f}%"
)

k3.metric(
    "Top Emitter",
    top_country
)

2026-07-23 16:28:16.675 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:16.677 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:16.677 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:16.678 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:16.679 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:16.680 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:16.681 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-23 16:28:16.682 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()